# Laboratorio 4 · Análisis exploratorio de datos (EDA)

**Universidad Técnica Nacional** · Minería de Datos ISW-911 · III Cuatrimestre 2026

**Integrantes del grupo**

* Jimena Jara
* Walbyn González
* Fiorella Lazo
* Seidy Alanis

**Conjunto de datos:** base de datos **original** del proyecto, *dunnhumby – The Complete Journey*
(las 8 tablas de origen, antes del proceso ETL y del Data Warehouse).

| Tabla | Contenido |
|---|---|
| `transaction_data` | Cada producto comprado en cada compra (línea de venta) |
| `product` | Departamento, marca, categoría y tamaño de cada producto |
| `hh_demographic` | Perfil demográfico de los hogares que lo completaron |
| `causal_data` | Exhibición en tienda y aparición en el folleto de cada producto, por tienda y semana |
| `campaign_table` | Hogares que recibieron cada campaña de mercadeo |
| `campaign_desc` | Tipo y duración (día de inicio y fin) de cada campaña |
| `coupon` | Productos que cubre cada cupón |
| `coupon_redempt` | Cupones canjeados por cada hogar y el día del canje |

Los archivos están comprimidos (`.xz`) en la carpeta `datos/` del repositorio público: https://github.com/walbyn504/Laboratorio4-Mineria

**Contenido**

1. Carga de las 8 tablas desde GitHub y construcción del conjunto de análisis
2. Características principales (funciones básicas)
3. Nombres de columnas en español
4. Valores nulos
5. Valores atípicos
6. Funciones de imputación
7. Conversión de tipos de datos
8. Conversión de variables categóricas a numéricas
9. Estandarización
10. Correlación
11. Variable dependiente y modelo candidato
12. Guardado de `data_process.csv`

## 1. Carga de las tablas desde GitHub

Todas las tablas se leen directamente desde el enlace *raw* del repositorio. Pandas descomprime los archivos `.xz` al leerlos.

`causal_data` tiene casi 37 millones de filas. Para no llenar la memoria de Colab se lee **por trozos** y de cada trozo
se guardan solo las filas que corresponden a una venta (mismo producto, tienda y semana).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

BASE = "https://raw.githubusercontent.com/walbyn504/Laboratorio4-Mineria/main/datos/"

transacciones = pd.read_csv(BASE + "transaction_data.csv.xz")
productos = pd.read_csv(BASE + "product.csv.xz")
hogares = pd.read_csv(BASE + "hh_demographic.csv.xz")
campanas_hogar = pd.read_csv(BASE + "campaign_table.csv.xz")
campanas = pd.read_csv(BASE + "campaign_desc.csv.xz")
cupones = pd.read_csv(BASE + "coupon.csv.xz")
canjes = pd.read_csv(BASE + "coupon_redempt.csv.xz")

In [ ]:
# causal_data: lectura por partes, conservando solo las combinaciones que tienen ventas
claves_venta = transacciones[["PRODUCT_ID", "STORE_ID", "WEEK_NO"]].drop_duplicates()

partes = []
filas_causal = 0
for trozo in pd.read_csv(BASE + "causal_data.csv.xz", chunksize=4_000_000,
                         dtype={"display": "string", "mailer": "string"}):
    filas_causal += len(trozo)
    partes.append(trozo.merge(claves_venta, on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"], how="inner"))
causal = pd.concat(partes, ignore_index=True)
del partes

print(f"causal_data: {filas_causal:,} filas en total, {len(causal):,} coinciden con alguna venta")

In [1]:
tablas = {
    "transaction_data": transacciones, "product": productos, "hh_demographic": hogares,
    "causal_data (filas con venta)": causal, "campaign_table": campanas_hogar,
    "campaign_desc": campanas, "coupon": cupones, "coupon_redempt": canjes,
}
display(pd.DataFrame({
    "filas": {nombre: tabla.shape[0] for nombre, tabla in tablas.items()},
    "columnas": {nombre: tabla.shape[1] for nombre, tabla in tablas.items()},
    "nulos": {nombre: int(tabla.isna().sum().sum()) for nombre, tabla in tablas.items()},
}))

for nombre, tabla in tablas.items():
    print(nombre)
    display(tabla.head(3))

NameError: name 'transacciones' is not defined

### Construcción del conjunto de análisis

Cada fila será **una línea de venta** (`transaction_data`). Se usa siempre `left join` para **no perder ninguna venta**:

| Tabla | Cómo se une | Columnas que aporta |
|---|---|---|
| `product` | por producto | departamento, marca, categoría, tamaño |
| `hh_demographic` | por hogar | edad, estado civil, ingreso, vivienda, composición, tamaño del hogar, hijos |
| `causal_data` | por producto + tienda + semana | tipo de exhibición y ubicación en el folleto |
| `campaign_table` + `campaign_desc` | por hogar, si el día de la compra está dentro de la campaña | cantidad de campañas activas del hogar ese día |
| `coupon` | por producto | si el producto tiene algún cupón |
| `coupon_redempt` + `coupon` | por hogar + día + producto | si el hogar canjeó un cupón de ese producto ese día |

De `product` no se usa `SUB_COMMODITY_DESC` (2 383 subcategorías, demasiado detalle) ni `MANUFACTURER` (es un código).

In [ ]:
llave_causal = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
duplicados_causal = causal.duplicated(llave_causal).sum()
print("Combinaciones repetidas en causal_data:", duplicados_causal)
causal = causal.drop_duplicates(llave_causal)

columnas_producto = ["PRODUCT_ID", "DEPARTMENT", "BRAND", "COMMODITY_DESC", "CURR_SIZE_OF_PRODUCT"]

ventas = (
    transacciones
    .merge(productos[columnas_producto], on="PRODUCT_ID", how="left", validate="many_to_one")
    .merge(hogares, on="household_key", how="left", validate="many_to_one")
    .merge(causal, on=llave_causal, how="left", validate="many_to_one")
)

# Campañas activas: el hogar recibió la campaña y el día de compra está entre su inicio y su fin
periodos = campanas_hogar[["household_key", "CAMPAIGN"]].merge(
    campanas[["CAMPAIGN", "START_DAY", "END_DAY"]], on="CAMPAIGN"
)
activas = np.zeros(len(ventas), dtype="int8")
for _, campana in periodos.groupby("CAMPAIGN"):
    inicio, fin = campana["START_DAY"].iloc[0], campana["END_DAY"].iloc[0]
    en_campana = ventas["household_key"].isin(campana["household_key"]) & ventas["DAY"].between(inicio, fin)
    activas += en_campana.to_numpy(dtype="int8")
ventas["ACTIVE_CAMPAIGNS"] = activas

# Cupones
ventas["PRODUCT_HAS_COUPON"] = ventas["PRODUCT_ID"].isin(cupones["PRODUCT_ID"]).astype("int8")

canjes_producto = (
    canjes.merge(cupones, on=["COUPON_UPC", "CAMPAIGN"])
    [["household_key", "DAY", "PRODUCT_ID"]]
    .drop_duplicates()
    .assign(COUPON_REDEEMED=1)
)
ventas = ventas.merge(canjes_producto, on=["household_key", "DAY", "PRODUCT_ID"], how="left")
ventas["COUPON_REDEEMED"] = ventas["COUPON_REDEEMED"].fillna(0).astype("int8")

# Comprobar que las uniones no perdieron ni duplicaron ventas
assert len(ventas) == len(transacciones)

del causal, claves_venta
print("Filas:", ventas.shape[0])
print("Columnas:", ventas.shape[1])
display(ventas.head())

## 2. Características principales del conjunto de datos

Se usan las funciones básicas de pandas: `shape`, `head`, `tail`, `sample`, `info`, `dtypes`,
`nunique`, `describe`, `duplicated` y `value_counts`.

In [ ]:
ventas.info()

In [ ]:
resumen_columnas = pd.DataFrame({
    "tipo": ventas.dtypes.astype(str),
    "valores_unicos": ventas.nunique(),
    "ejemplo": ventas.iloc[0],
})
display(resumen_columnas)

In [ ]:
display(ventas.tail())
display(ventas.sample(5, random_state=42))

In [ ]:
# Estadísticas de las variables numéricas
display(ventas.describe().round(2))

In [ ]:
# Estadísticas de las variables de texto
display(ventas.describe(exclude="number"))

In [ ]:
print("Filas completamente duplicadas:", ventas.duplicated().sum())
print("Hogares distintos:", ventas["household_key"].nunique())
print("Compras (canastas) distintas:", ventas["BASKET_ID"].nunique())
print("Productos distintos vendidos:", ventas["PRODUCT_ID"].nunique())
print("Días cubiertos:", ventas["DAY"].min(), "a", ventas["DAY"].max())

In [ ]:
from matplotlib.ticker import FuncFormatter

en_miles = FuncFormatter(lambda x, _: f"{x / 1000:,.0f}")


def barras_con_etiquetas(serie, eje, titulo, color="steelblue"):
    serie.plot.bar(ax=eje, rot=0, color=color)
    eje.set_title(titulo)
    eje.set_xlabel("")
    eje.set_ylabel("miles de líneas")
    eje.yaxis.set_major_formatter(en_miles)
    eje.bar_label(eje.containers[0], fontsize=8, padding=3,
                  labels=[f"{v / 1000:,.0f} mil" for v in serie])
    eje.margins(y=0.12)


fig, ejes = plt.subplots(2, 2, figsize=(15, 9))

ventas["DEPARTMENT"].value_counts().head(10).sort_values().plot.barh(ax=ejes[0, 0])
ejes[0, 0].set_title("Top 10 departamentos")
ejes[0, 0].set_ylabel("")
ejes[0, 0].set_xlabel("miles de líneas")
ejes[0, 0].xaxis.set_major_formatter(en_miles)

barras_con_etiquetas(ventas["BRAND"].value_counts(), ejes[0, 1], "Tipo de marca")

orden_edad = ["19-24", "25-34", "35-44", "45-54", "55-64", "65+"]
barras_con_etiquetas(ventas["AGE_DESC"].value_counts().reindex(orden_edad), ejes[1, 0],
                     "Edad del jefe de hogar (solo hogares con perfil)")

barras_con_etiquetas(ventas["ACTIVE_CAMPAIGNS"].value_counts().sort_index(), ejes[1, 1],
                     "Campañas activas del hogar el día de la compra")

fig.suptitle("Líneas de venta por variable", fontsize=14)
plt.tight_layout()
plt.show()

**Hallazgos.** El conjunto tiene 2 595 732 líneas de venta de 2 500 hogares, 276 484 compras y 92 339 productos distintos durante 711 días. No hay filas duplicadas.

* El departamento `GROCERY` concentra la mayoría de las ventas y casi tres de cada cuatro líneas son de marca nacional.
* De los 36.8 millones de registros de `causal_data`, solo 483 935 coinciden con alguna venta; el resto son productos exhibidos o anunciados en semanas en que nadie del estudio los compró. En `causal_data` hay 615 combinaciones producto-tienda-semana repetidas, que se eliminaron antes de unir para no duplicar ventas.
* El 70 % de las líneas ocurre cuando el hogar no tiene ninguna campaña activa; el resto tiene entre 1 y 5 campañas a la vez.
* Las columnas demográficas y las de `causal_data` tienen muchos menos datos que el resto: se analizan en la sección 4.

## 3. Nombres de columnas en español

In [ ]:
nombres_en_espanol = {
    "household_key": "id_hogar",
    "BASKET_ID": "id_compra",
    "DAY": "dia",
    "PRODUCT_ID": "id_producto",
    "QUANTITY": "cantidad",
    "SALES_VALUE": "importe_venta",
    "STORE_ID": "id_tienda",
    "RETAIL_DISC": "descuento_tienda",
    "TRANS_TIME": "hora_transaccion",
    "WEEK_NO": "semana",
    "COUPON_DISC": "descuento_cupon",
    "COUPON_MATCH_DISC": "igualacion_descuento_cupon",
    "DEPARTMENT": "departamento",
    "BRAND": "tipo_marca",
    "COMMODITY_DESC": "categoria",
    "CURR_SIZE_OF_PRODUCT": "tamano_producto",
    "AGE_DESC": "edad",
    "MARITAL_STATUS_CODE": "estado_civil",
    "INCOME_DESC": "ingreso",
    "HOMEOWNER_DESC": "tipo_vivienda",
    "HH_COMP_DESC": "composicion_hogar",
    "HOUSEHOLD_SIZE_DESC": "tamano_hogar",
    "KID_CATEGORY_DESC": "cantidad_hijos",
    "display": "exhibicion",
    "mailer": "folleto",
    "ACTIVE_CAMPAIGNS": "campanas_activas",
    "PRODUCT_HAS_COUPON": "producto_con_cupon",
    "COUPON_REDEEMED": "cupon_canjeado",
}

ventas = ventas.rename(columns=nombres_en_espanol)
print(list(ventas.columns))
display(ventas.head())

## 4. Valores nulos

En la base original los faltantes aparecen de dos formas:

* **`NaN`**: al unir las tablas quedan vacíos los hogares sin perfil demográfico y las ventas sin registro en `causal_data`.
* **Textos en blanco** (`" "`): en `product`, algunos productos tienen el departamento, la categoría o el tamaño
  escritos como un espacio. Pandas los lee como texto válido, así que primero se convierten a `NaN`.

In [ ]:
def es_blanco(serie):
    return serie.astype("string").str.strip().eq("").fillna(False).astype(bool)


columnas_texto = ventas.select_dtypes(exclude="number").columns
blancos = {columna: int(es_blanco(ventas[columna]).sum()) for columna in columnas_texto}
print("Textos en blanco por columna:", {c: n for c, n in blancos.items() if n > 0})

for columna in columnas_texto:
    ventas[columna] = ventas[columna].mask(es_blanco(ventas[columna]))

In [ ]:
nulos = pd.DataFrame({
    "nulos": ventas.isna().sum(),
    "porcentaje": (100 * ventas.isna().mean()).round(2),
})
display(nulos.sort_values("porcentaje", ascending=False))
print("Total de valores nulos:", nulos["nulos"].sum())

In [ ]:
porcentajes = nulos["porcentaje"].sort_values()
colores = ["tomato" if p > 0 else "lightgray" for p in porcentajes]

ax = porcentajes.plot.barh(figsize=(9, 9), color=colores)
ax.set_title("Porcentaje de valores nulos por columna")
ax.set_xlabel("% de líneas de venta")
ax.set_xlim(0, 100)
for barra, valor in zip(ax.containers[0], porcentajes):
    if valor > 0:
        ax.text(valor + 1, barra.get_y() + barra.get_height() / 2, f"{valor:.1f} %", va="center")
plt.tight_layout()
plt.show()

El siguiente gráfico agrupa las líneas de venta según **qué grupo de datos les falta**.

In [ ]:
grupos = {
    "perfil del hogar": ["edad", "estado_civil", "ingreso", "tipo_vivienda",
                         "composicion_hogar", "tamano_hogar", "cantidad_hijos"],
    "datos causales": ["exhibicion", "folleto"],
    "tamaño del producto": ["tamano_producto"],
    "clasificación del producto": ["departamento", "categoria"],
}
falta = pd.DataFrame({grupo: ventas[columnas].isna().any(axis=1) for grupo, columnas in grupos.items()})


def describir_patron(fila):
    faltan = [grupo for grupo, valor in fila.items() if valor]
    return "Datos completos" if not faltan else "Falta: " + " + ".join(faltan)


patrones = falta.value_counts()
patrones.index = [describir_patron(pd.Series(p, index=falta.columns)) for p in patrones.index]
patrones = patrones.sort_values().tail(8)

ax = patrones.plot.barh(figsize=(12, 5),
                        color=["seagreen" if i == "Datos completos" else "tomato" for i in patrones.index])
ax.set_title("¿Qué datos le faltan a cada línea de venta? (8 combinaciones más frecuentes)")
ax.set_xlabel("miles de líneas")
ax.xaxis.set_major_formatter(en_miles)
ax.bar_label(ax.containers[0], padding=4,
             labels=[f"{v:,.0f}  ({100 * v / len(ventas):.1f} %)" for v in patrones])
ax.margins(x=0.4)
plt.tight_layout()
plt.show()

for grupo in ["perfil del hogar", "datos causales"]:
    faltantes_grupo = ventas[grupos[grupo]].isna()
    print(f"Las columnas de '{grupo}' faltan en las mismas filas:",
          faltantes_grupo.all(axis=1).equals(faltantes_grupo.any(axis=1)))

**Hallazgos.**

* **Datos causales (78 %).** `exhibicion` y `folleto` faltan en 2 031 958 líneas y **siempre juntas**: esas ventas no tienen registro en `causal_data` para ese producto, tienda y semana.
* **Perfil del hogar (45 %).** Las 7 columnas demográficas faltan en 1 168 429 líneas y **siempre juntas**: solo 801 de los 2 500 hogares completaron el perfil.
* **Tamaño del producto (18 %).** Venía en blanco en el origen. Es texto libre con más de 4 000 formatos distintos ("16 OZ", "1 LB", "12 CT"…).
* **Departamento y categoría (0.3 %).** 7 839 líneas de productos sin clasificación en el catálogo.
* Solo el **9.6 %** de las líneas tiene todos los datos completos. Las columnas de `transaction_data` (cantidades, importes, descuentos) no tienen nulos.
* El tratamiento se define en la sección 6.

## 5. Valores atípicos

Se usa el criterio del rango intercuartílico (IQR): un valor es atípico si está por debajo de
Q1 − 1.5·IQR o por encima de Q3 + 1.5·IQR. Es el mismo criterio que dibujan los diagramas de caja.
También se revisan registros inconsistentes (cantidad 0).

In [ ]:
columnas_numericas = [
    "cantidad",
    "importe_venta",
    "descuento_tienda",
    "descuento_cupon",
    "igualacion_descuento_cupon",
]


def resumen_atipicos_iqr(tabla, columnas):
    """Devuelve una tabla con los límites IQR y la cantidad de atípicos por columna."""
    filas = []
    for columna in columnas:
        valores = tabla[columna].dropna()
        q1, q3 = valores.quantile([0.25, 0.75])
        iqr = q3 - q1
        inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        atipicos = ((valores < inferior) | (valores > superior)).sum()
        filas.append({
            "columna": columna, "Q1": q1, "Q3": q3, "IQR": iqr,
            "limite_inferior": inferior, "limite_superior": superior,
            "atipicos": atipicos,
            "porcentaje": round(100 * atipicos / len(valores), 2),
        })
    return pd.DataFrame(filas).set_index("columna")


display(resumen_atipicos_iqr(ventas, columnas_numericas).round(2))

In [ ]:
fig, ejes = plt.subplots(1, len(columnas_numericas), figsize=(16, 5))

for columna, eje in zip(columnas_numericas, ejes):
    valores = ventas[columna].dropna()
    q1, q3 = valores.quantile([0.25, 0.75])
    iqr = q3 - q1
    extremos = valores[(valores < q1 - 1.5 * iqr) | (valores > q3 + 1.5 * iqr)]

    eje.boxplot(valores, showfliers=False)
    eje.scatter([1] * extremos.nunique(), extremos.unique(), s=8, alpha=0.35, color="tomato")
    eje.set_yscale("symlog", linthresh=1)
    eje.set_title(columna, fontsize=10)
    eje.set_xticks([])

fig.suptitle("Diagramas de caja (escala symlog) · atípicos en rojo")
plt.tight_layout()
plt.show()

In [ ]:
columnas_revision = ["id_compra", "departamento", "categoria", "cantidad", "importe_venta"]
display(ventas.nlargest(10, "cantidad")[columnas_revision])
display(ventas.nlargest(10, "importe_venta")[columnas_revision])

In [ ]:
# ¿En qué departamentos están las cantidades extremas?
cantidad_por_depto = (
    ventas.groupby("departamento")["cantidad"]
    .agg(["median", "max"])
    .sort_values("max", ascending=False)
    .head(8)
)
display(cantidad_por_depto)

ax = cantidad_por_depto["max"].sort_values().plot.barh(figsize=(8, 4), logx=True)
ax.set_title("Cantidad máxima por departamento (escala log)")
ax.set_xlabel("cantidad máxima")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
revision = pd.Series({
    "Cantidad 0 (error)": (ventas["cantidad"] == 0).sum(),
    "Importe 0 (descuento total, válido)": (ventas["importe_venta"] == 0).sum(),
})
display(revision.to_frame("lineas"))

ax = revision.plot.barh(figsize=(9, 2.8), color=["tomato", "seagreen"])
ax.set_title("Registros revisados por lógica del negocio")
ax.set_xlabel("líneas de venta")
ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=4)
ax.margins(x=0.2)
plt.tight_layout()
plt.show()

**Hallazgos.**

* En `cantidad`, Q1 = Q3 = 1: casi todas las ventas son de 1 unidad, así que el IQR es 0 y cualquier valor
  distinto de 1 sale como atípico. Por eso, para el tratamiento se usan percentiles (sección 6).
* Las cantidades extremas (hasta 89 638) vienen de `KIOSK-GAS` y `MISC SALES TRAN`: en combustible la cantidad no
  se mide en unidades como el resto de productos.
* En los descuentos la mayoría de valores es 0, así que el IQR marca como atípico cualquier descuento.
  Son valores reales y se conservan.
* Hay 14 466 líneas con **cantidad 0**: una venta no puede tener 0 unidades, se tratarán como faltantes.
* Hay unas 18 850 líneas con **importe 0**: productos con descuento total. Son válidas y se conservan.

## 6. Funciones de imputación

Decisiones y justificación:

| Hallazgo | Tratamiento | Justificación |
|---|---|---|
| 7 columnas demográficas nulas (45 %) | Categoría constante **"Desconocido"** + indicador `tiene_perfil` (1/0) | Falta casi la mitad. Imputar la moda diría que el 45 % de los hogares son, por ejemplo, de 45-54 años, y eso sesgaría las variables. El indicador permite al modelo distinguir los hogares sin perfil. |
| `exhibicion` y `folleto` nulos (sin registro en `causal_data`) | Categoría constante **"Sin dato"** + indicador `tiene_dato_causal` (1/0) | `causal_data` solo registra algunos productos, tiendas y semanas. Imputar la moda ("No exhibido") afirmaría algo que no se sabe. |
| `departamento` y `categoria` nulos (0.3 %) | Categoría constante **"SIN CLASIFICAR"** | Son pocos, pero imputar la moda (`GROCERY`) mezclaría productos sin clasificar con abarrotes. |
| `tamano_producto` nulo (18 %) | **Se elimina la columna** | Es texto libre con más de 4 000 formatos y unidades distintas (OZ, LB, CT); no se puede comparar ni imputar de forma confiable. |
| `cantidad = 0` (14 466) | Convertir a `NaN` e imputar con la **mediana de su categoría** | Una venta no puede tener 0 unidades. La mediana por categoría respeta las unidades de cada producto y no la afectan los extremos. |
| `cantidad` extrema | **Recorte (winsorización) a los percentiles 1 y 99** | El IQR es 0 en esta columna. Recortar limita el efecto del combustible sin borrar filas. |
| `importe_venta` y descuentos extremos | **Se conservan** | Son valores reales. `importe_venta` es la variable objetivo y recortarla sesgaría el modelo. |

In [ ]:
def imputar_categorica(tabla, columna, estrategia="constante", valor="Desconocido"):
    """Imputa los NaN de una columna categórica.

    estrategia = "constante": reemplaza por `valor`.
    estrategia = "moda": reemplaza por el valor más frecuente.
    Devuelve la cantidad de valores imputados.
    """
    faltantes = tabla[columna].isna()
    relleno = tabla[columna].mode()[0] if estrategia == "moda" else valor
    tabla[columna] = tabla[columna].fillna(relleno)
    return int(faltantes.sum())


def imputar_mediana_por_grupo(tabla, columna, grupo):
    """Imputa los NaN de una columna numérica con la mediana de su grupo.

    Si un grupo no tiene ningún valor, se usa la mediana global.
    """
    faltantes = tabla[columna].isna()
    if not faltantes.any():
        return 0
    medianas = tabla.groupby(grupo, observed=True)[columna].transform("median")
    tabla[columna] = tabla[columna].fillna(medianas).fillna(tabla[columna].median())
    return int(faltantes.sum())


def recortar_atipicos(tabla, columna, inferior=0.01, superior=0.99):
    """Winsoriza una columna: lleva los valores fuera de los percentiles a esos límites."""
    limite_inf, limite_sup = tabla[columna].quantile([inferior, superior])
    fuera = ((tabla[columna] < limite_inf) | (tabla[columna] > limite_sup)).sum()
    tabla[columna] = tabla[columna].clip(limite_inf, limite_sup)
    return int(fuera), limite_inf, limite_sup

In [ ]:
nulos_antes = ventas.isna().sum()

# 1) Perfil demográfico: indicador + "Desconocido"
columnas_demograficas = grupos["perfil del hogar"]
ventas["tiene_perfil"] = ventas["edad"].notna().astype("int8")
for columna in columnas_demograficas:
    n = imputar_categorica(ventas, columna, valor="Desconocido")
print(f"Perfil demográfico: {n:,} líneas imputadas con 'Desconocido' en {len(columnas_demograficas)} columnas")

# 2) Datos causales: indicador + "Sin dato"
ventas["tiene_dato_causal"] = ventas["exhibicion"].notna().astype("int8")
for columna in ["exhibicion", "folleto"]:
    n = imputar_categorica(ventas, columna, valor="Sin dato")
    print(f"{columna}: {n:,} valores imputados con 'Sin dato'")

# 3) Producto sin clasificar
for columna in ["departamento", "categoria"]:
    n = imputar_categorica(ventas, columna, valor="SIN CLASIFICAR")
    print(f"{columna}: {n:,} valores imputados con 'SIN CLASIFICAR'")

# 4) Tamaño del producto: se elimina
ventas = ventas.drop(columns="tamano_producto")
print("tamano_producto: columna eliminada")

# 5) Cantidad 0 -> NaN -> mediana de su categoría
ventas["cantidad"] = ventas["cantidad"].mask(ventas["cantidad"] == 0)
nulos_antes["cantidad"] = ventas["cantidad"].isna().sum()
n = imputar_mediana_por_grupo(ventas, "cantidad", "categoria")
print(f"cantidad: {n:,} valores en 0 imputados con la mediana de su categoría")

# 6) Atípicos extremos de cantidad -> recorte P1-P99
n, li, ls = recortar_atipicos(ventas, "cantidad")
print(f"cantidad: {n:,} valores recortados al rango [{li:.0f}, {ls:.0f}]")

nulos_despues = ventas.isna().sum().reindex(nulos_antes.index, fill_value=0)
print("Nulos restantes en el conjunto:", ventas.isna().sum().sum())

In [ ]:
comparacion = pd.DataFrame({"antes": nulos_antes, "después": nulos_despues})
comparacion = comparacion[comparacion.sum(axis=1) > 0].drop(index="tamano_producto")

ax = comparacion.plot.barh(figsize=(9, 5.5), color=["tomato", "seagreen"])
ax.set_title("Valores faltantes antes y después de imputar")
ax.set_xlabel("miles de líneas")
ax.xaxis.set_major_formatter(en_miles)
plt.tight_layout()
plt.show()

## 7. Conversión de tipos de datos

* La base original no tiene fecha de calendario, solo el número de `dia` (1 a 711) y la `semana`. Se dejan como enteros.
* `hora_transaccion` viene como un número HHMM (por ejemplo 1631 = 16:31). Se extrae la `hora` (0–23).
* `exhibicion` y `folleto` vienen como códigos de una letra o número. Se traducen a su significado según la
  guía de usuario de dunnhumby.
* Los identificadores (`id_hogar`, `id_compra`, `id_producto`, `id_tienda`) son códigos, no cantidades:
  se convierten a texto para que no entren en cálculos numéricos.
* Las variables de texto con pocos valores distintos se convierten a `category`, lo que además reduce memoria.
* `cantidad` vuelve a entero después de la imputación.

In [ ]:
memoria_antes = ventas.memory_usage(deep=True).sum() / 1024**2

codigos_exhibicion = {
    "0": "No exhibido", "1": "Frente de tienda", "2": "Fondo de tienda",
    "3": "Cabecera frontal", "4": "Cabecera de pasillo central", "5": "Cabecera trasera",
    "6": "Cabecera de pasillo lateral", "7": "En el pasillo", "9": "Ubicación secundaria",
    "A": "En el estante", "Sin dato": "Sin dato",
}
codigos_folleto = {
    "0": "No anunciado", "A": "Destacado página interior", "C": "Línea en página interior",
    "D": "Destacado portada", "F": "Destacado contraportada", "H": "Destacado envoltura frontal",
    "J": "Cupón envoltura interior", "L": "Destacado envoltura trasera", "P": "Cupón página interior",
    "X": "Gratis en página interior", "Z": "Gratis en portada o contraportada", "Sin dato": "Sin dato",
}
ventas["exhibicion"] = ventas["exhibicion"].astype(str).map(codigos_exhibicion)
ventas["folleto"] = ventas["folleto"].astype(str).map(codigos_folleto)
assert ventas[["exhibicion", "folleto"]].notna().all().all(), "Hay un código sin traducir"

ventas["hora"] = (ventas["hora_transaccion"] // 100).astype("int8")

for columna in ["id_hogar", "id_compra", "id_producto", "id_tienda"]:
    ventas[columna] = ventas[columna].astype("string")

columnas_categoricas = ["departamento", "tipo_marca", "categoria", "exhibicion", "folleto"] + columnas_demograficas
for columna in columnas_categoricas:
    ventas[columna] = ventas[columna].astype("category")

ventas["cantidad"] = ventas["cantidad"].round().astype("int64")
ventas["dia"] = ventas["dia"].astype("int16")
ventas["semana"] = ventas["semana"].astype("int8")

memoria_despues = ventas.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria: {memoria_antes:,.0f} MB -> {memoria_despues:,.0f} MB")
display(ventas.dtypes.to_frame("tipo"))

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))
for columna, eje in zip(["exhibicion", "folleto"], ejes):
    conteo = ventas.loc[ventas[columna] != "Sin dato", columna].value_counts().sort_values()
    conteo = conteo[conteo > 0]
    conteo.plot.barh(ax=eje)
    eje.set_title(f"{columna} (solo ventas con dato causal)")
    eje.set_xlabel("miles de líneas")
    eje.set_ylabel("")
    eje.xaxis.set_major_formatter(en_miles)
plt.tight_layout()
plt.show()

## 8. Conversión de variables categóricas a numéricas

| Variable | Valores | Técnica |
|---|---|---|
| `tipo_marca` | National / Private | Binaria 0/1 |
| `exhibicion`, `folleto` | Exhibido o anunciado / no | Binarias `en_exhibicion` y `en_folleto` (1 si el producto estaba exhibido o en el folleto) |
| `edad`, `ingreso`, `tamano_hogar`, `cantidad_hijos` | Rangos con orden natural | **Ordinal**: 1, 2, 3… según el orden (Desconocido = 0, ya marcado en `tiene_perfil`) |
| `estado_civil`, `tipo_vivienda`, `composicion_hogar`, `exhibicion`, `folleto` | 4 a 12 categorías sin orden | **One-hot** (`get_dummies`) |
| `departamento`, `categoria` | 44 y 308 | **Codificación por etiqueta** (`cat.codes`): one-hot crearía más de 350 columnas para 2.6 millones de filas, y el árbol de regresión propuesto trabaja bien con códigos |

`campanas_activas`, `producto_con_cupon` y `cupon_canjeado` ya son numéricas.

In [ ]:
codificadas = pd.DataFrame(index=ventas.index)

codificadas["es_marca_propia"] = (ventas["tipo_marca"] == "Private").astype("int8")
codificadas["en_exhibicion"] = (~ventas["exhibicion"].isin(["No exhibido", "Sin dato"])).astype("int8")
codificadas["en_folleto"] = (~ventas["folleto"].isin(["No anunciado", "Sin dato"])).astype("int8")

ordenes = {
    "edad": ["19-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "ingreso": ["Under 15K", "15-24K", "25-34K", "35-49K", "50-74K", "75-99K",
                "100-124K", "125-149K", "150-174K", "175-199K", "200-249K", "250K+"],
    "tamano_hogar": ["1", "2", "3", "4", "5+"],
    "cantidad_hijos": ["None/Unknown", "1", "2", "3+"],
}
for columna, orden in ordenes.items():
    mapa = {valor: posicion for posicion, valor in enumerate(orden, start=1)}
    mapa["Desconocido"] = 0
    codificadas[f"{columna}_num"] = ventas[columna].astype(str).map(mapa).astype("int8")

codificadas["departamento_cod"] = ventas["departamento"].cat.codes.astype("int16")
codificadas["categoria_cod"] = ventas["categoria"].cat.codes.astype("int16")

indicadoras = pd.get_dummies(
    ventas[["estado_civil", "tipo_vivienda", "composicion_hogar", "exhibicion", "folleto"]],
    dtype="int8",
)
indicadoras.columns = (
    indicadoras.columns.str.lower()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
codificadas = pd.concat([codificadas, indicadoras], axis=1)

print("Columnas numéricas creadas:", codificadas.shape[1])
display(codificadas.head())

## 9. Estandarización

Se aplica estandarización **z-score** (media 0, desviación estándar 1) a los predictores numéricos continuos.
Se eligió estandarizar en vez de normalizar (min-max) porque los descuentos y la cantidad tienen valores
extremos: con min-max casi todos los datos quedarían comprimidos cerca de 0.
La variable objetivo `importe_venta` no se estandariza para que el resultado del modelo quede en la unidad original.

In [ ]:
def estandarizar_z(tabla, columnas):
    """Devuelve las columnas estandarizadas (x - media) / desviación, con sufijo _z."""
    datos = tabla[columnas].astype("float64")
    return ((datos - datos.mean()) / datos.std(ddof=0)).add_suffix("_z")


columnas_a_estandarizar = [
    "cantidad", "descuento_tienda", "descuento_cupon", "igualacion_descuento_cupon", "campanas_activas",
]
estandarizadas = estandarizar_z(ventas, columnas_a_estandarizar)

display(pd.DataFrame({
    "media": estandarizadas.mean(),
    "desviacion_estandar": estandarizadas.std(ddof=0),
}).round(3))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ventas["descuento_tienda"].plot.hist(bins=60, ax=ejes[0], log=True)
ejes[0].set_title("descuento_tienda (original)")
estandarizadas["descuento_tienda_z"].plot.hist(bins=60, ax=ejes[1], log=True, color="seagreen")
ejes[1].set_title("descuento_tienda_z (estandarizado)")
plt.tight_layout()
plt.show()

In [ ]:
ventas_procesadas = pd.concat([ventas, codificadas, estandarizadas], axis=1)
del codificadas, estandarizadas
print("Filas y columnas:", ventas_procesadas.shape)

## 10. Correlación

Se calcula la correlación de Pearson entre la variable objetivo y los predictores numéricos y codificados.

In [ ]:
columnas_correlacion = [
    "importe_venta", "cantidad", "descuento_tienda", "descuento_cupon",
    "igualacion_descuento_cupon", "es_marca_propia", "en_exhibicion", "en_folleto",
    "tiene_dato_causal", "campanas_activas", "producto_con_cupon", "cupon_canjeado",
    "tiene_perfil", "edad_num", "ingreso_num", "tamano_hogar_num", "hora", "semana",
]
correlacion = ventas_procesadas[columnas_correlacion].corr(method="pearson")

fig, ax = plt.subplots(figsize=(13, 11))
imagen = ax.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
posiciones = range(len(columnas_correlacion))
ax.set_xticks(posiciones, columnas_correlacion, rotation=45, ha="right")
ax.set_yticks(posiciones, columnas_correlacion)
for i in posiciones:
    for j in posiciones:
        valor = correlacion.iloc[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(valor) > 0.65 else "black")
ax.set_title("Matriz de correlación de Pearson")
fig.colorbar(imagen, ax=ax, label="Correlación")
plt.tight_layout()
plt.show()

In [ ]:
con_objetivo = correlacion["importe_venta"].drop("importe_venta").sort_values()
display(con_objetivo.round(3).to_frame("correlacion_con_importe_venta"))

ax = con_objetivo.plot.barh(
    figsize=(8, 6), color=np.where(con_objetivo < 0, "tomato", "steelblue")
)
ax.set_title("Correlación de cada variable con importe_venta")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Hallazgos.**

* `cantidad` es la variable con mayor relación con `importe_venta` (**r ≈ 0.48**, positiva y moderada): a más unidades, mayor importe.
* `descuento_tienda` tiene **r ≈ −0.25**. Los descuentos se registran como números negativos, así que las líneas con mayor descuento suelen tener un importe más alto (productos más caros).
* Las variables de las tablas nuevas (exhibición, folleto, campañas y cupones) tienen correlación casi nula con el importe (|r| < 0.05). Sí se relacionan entre ellas: `en_folleto` con `tiene_dato_causal` (r ≈ 0.83) y con `descuento_tienda` (r ≈ −0.33): los productos anunciados en el folleto suelen tener descuento.
* `campanas_activas` se relaciona con `semana` (r ≈ 0.42) porque la mayoría de campañas ocurren en la segunda mitad del periodo.
* Las variables demográficas están **muy correlacionadas entre sí** (r entre 0.65 y 0.88) y con `tiene_perfil`. Gran parte de esa relación es artificial: el 45 % de las líneas tiene el código 0 ("Desconocido") en todas ellas a la vez. Para un modelo lineal habría multicolinealidad; conviene usar solo algunas o el árbol de regresión, que no se ve afectado.
* Que la relación lineal sea débil no descarta relaciones no lineales o por categoría de producto, que es la razón para proponer un árbol de regresión.

## 11. Variable dependiente y modelo candidato

**Variable dependiente:** `importe_venta`, el monto cobrado en cada línea de venta.

**Tipo de problema:** como `importe_venta` es una variable **numérica continua**, el problema es de **regresión**
(no de clasificación).

**Modelo propuesto: árbol de regresión.**

* La correlación lineal de cada variable con el importe es débil o moderada (sección 10). Un modelo lineal
  no captaría bien relaciones que cambian según el producto. El árbol, en cambio, divide los datos por
  categoría, departamento, exhibición, folleto o perfil del hogar y estima el importe dentro de cada grupo.
* Trabaja bien con las variables codificadas por etiqueta (`categoria_cod`, `departamento_cod`) y no
  necesita que los datos estén estandarizados.
* No le afecta la multicolinealidad entre las variables demográficas (sección 10).
* Es interpretable: se pueden leer las reglas que llevan a cada predicción.

**Modelo de referencia:** una **regresión lineal múltiple** con las variables estandarizadas, para comparar
el error del árbol contra un modelo simple.

**Precaución:** los descuentos y el canje de cupones se registran en el mismo momento de la venta. Si el objetivo es
predecir el importe *antes* de la compra, habría que excluirlos para no filtrar información del resultado.

## 12. Guardado del conjunto de datos procesado

Se guarda el resultado con todas las actividades del EDA aplicadas: nombres en español, nulos imputados,
atípicos tratados, tipos convertidos, variables codificadas y estandarizadas.

In [ ]:
ventas_procesadas.to_csv("data_process.csv", index=False)

import os
print("Archivo guardado: data_process.csv")
print("Filas y columnas:", ventas_procesadas.shape)
print(f"Tamaño: {os.path.getsize('data_process.csv') / 1024**2:,.0f} MB")
display(pd.read_csv("data_process.csv", nrows=5))

# Para descargarlo desde Colab (opcional, el archivo es grande):
# from google.colab import files
# files.download("data_process.csv")